# Traffic Accident Risk Analysis Using Databricks and PySpark

In this project, we build an end-to-end big data pipeline using Databricks and PySpark to analyze traffic accident risk.

This notebook covers:

1. **Environment setup** — create catalog and schema
2. **Bronze layer** — generate mock raw traffic data
3. **Silver layer** — clean and standardize the data
4. **Gold layer** — join and aggregate data for analysis
5. **EDA** — explore accident patterns and trends
6. **Dashboard-ready outputs** — prepare summary tables for visualization

## 1. Environment Setup — Create Catalog and Schema

In this section, we are creating the Databricks catalog and schema used to store all Bronze, Silver, and Gold tables for this project.

In [0]:
%sql
-- Create Catalog
CREATE CATALOG IF NOT EXISTS traffic_catalog;

-- Create Schema
CREATE SCHEMA IF NOT EXISTS traffic_catalog.traffic_schema;

## 2. Bronze Layer — Mock Data Generation

The Bronze layer stores raw data as-is.  
For this project, we are generating realistic mock traffic accident data with intentional quality issues such as:

- missing values
- inconsistent text formatting
- invalid values
- incorrect data types

These issues help simulate real-world raw datasets.

### 2.1 Helper Functions

The helper functions below are used to simulate realistic data quality issues in the Bronze layer. These include missing values, inconsistent text formatting, and invalid entries. This helps demonstrate how the Silver layer cleans and standardizes raw data.

In [0]:
# Import required PySpark functions and Python libraries
from pyspark.sql.functions import *
import random
from datetime import datetime, timedelta

# Set seed for reproducibility so results are consistent each time
random.seed(42)

random_choice = random.choice

# Randomly return None to simulate missing values
def random_nullable(value, chance=0.1):
    return None if random.random() < chance else value

# Randomly change text format to simulate inconsistent capitalization
def random_string_case(text):
    return random.choice([text.lower(), text.upper(), text.title()])

# Randomly insert invalid values to simulate bad data quality
def random_invalid(value):
    return random.choice([-1, None, value])

### 2.2 Create Locations Bronze Table

This table stores road and city information for each location, including city name, road type, and speed limit.

In [0]:
locations_data = []

cities = ["Toronto", "Brampton", "Burlington", "Mississauga"]
road_types = ["Highway", "Urban", "Rural"]

for i in range(1, 501):
    locations_data.append((i,
                           random_string_case(random_choice(cities)),
                           random_string_case(random_choice(road_types)),
                           random_choice([40, 50, 60, 80, 100])
    ))

spark.sql("DROP TABLE IF EXISTS traffic_catalog.traffic_schema.locations_bronze")
locations_df = spark.createDataFrame(locations_data,
                                      ["location_id", "city", "road_type", "speed_limit"])

locations_df.write.format("delta").mode("overwrite").saveAsTable(
                                                            "traffic_catalog.traffic_schema.locations_bronze"
                                                            )                          

This raw table stores location-related information such as city, road type, and speed limit. Some values are intentionally inconsistent or invalid to simulate real-world raw data.

### 2.3 Create Weather Bronze Table

This table stores environmental conditions such as weather type, visibility, and temperature. Some temperature values are intentionally stored using the wrong data type to simulate raw data issues.

In [0]:
weather_data = []

conditions = ["Clear", "Rain", "Snow", "Fog"]

for i in range(1, 501):
    weather_data.append((i,
                         random_string_case(random_choice(conditions)),
                         random_nullable(random.uniform(0.5, 10.0)),
                         str(random_nullable(random.uniform(-20, 35))) # intentionally stored as string(wrong datatype)
                         ))
    
spark.sql("DROP TABLE IF EXISTS traffic_catalog.traffic_schema.weather_bronze")
weather_df = spark.createDataFrame(weather_data, ["weather_id", "condition", "visibility", "temperature"])
    
weather_df.write.format("delta").mode("overwrite").saveAsTable("traffic_catalog.traffic_schema.weather_bronze")

This raw table stores weather conditions such as visibility and temperature. We intentionally include missing and inconsistent values for later cleaning.

### 2.4 Create Vehicles Bronze Table

This table stores vehicle and driver details such as vehicle type, driver age, and driver gender.

In [0]:
vehicles_data = []

vehicle_types = ["Car", "Truck", "Bike"]
genders = ["Male", "Female", "Other"]

for i in range(1, 501):
    vehicles_data.append((i,
                          random_string_case(random_choice(vehicle_types)),
                          random_invalid(random.randint(16,80)),
                          random_nullable(random_choice(genders))
                          ))
    
spark.sql("DROP TABLE IF EXISTS traffic_catalog.traffic_schema.vehicles_bronze")    
vehicles_df = spark.createDataFrame(vehicles_data, ["vehicle_id", "vehicle_type", "driver_age", "driver_gender"])
vehicles_df.write.format("delta").mode("overwrite").saveAsTable("traffic_catalog.traffic_schema.vehicles_bronze")

This table stores driver and vehicle information. Some records contain missing gender values or unrealistic ages to demonstrate cleaning in the Silver layer.

### 2.5 Create Accidents Bronze Table

This is the main fact table. It links to locations, weather, and vehicles, and stores accident severity.

In [0]:
accidents_data = []
start_date = datetime(2025, 1, 1)

for i in range(1, 501):
    accident_dt = start_date + timedelta(days=random.randint(0, 364), hours=random.randint(0, 23), minutes=random.randint(0, 59))
    accidents_data.append((
        i,
        random.randint(1, 100),
        random.randint(1, 100),
        random.randint(1, 200),
        accident_dt.strftime("%Y-%m-%d"),
        accident_dt.strftime("%H:%M:%S"),
        random_invalid(random.randint(1, 5))
    ))

spark.sql("DROP TABLE IF EXISTS traffic_catalog.traffic_schema.accidents_bronze")
accidents_df = spark.createDataFrame(accidents_data, ["accident_id", "location_id", "weather_id", "vehicle_id", "accident_date", "accident_time","severity"])
accidents_df.write.format("delta").mode("overwrite").saveAsTable("traffic_catalog.traffic_schema.accidents_bronze")

This is the main fact table containing accident details such as date, time, severity, and foreign keys linking to other tables.

### 2.6 Create Violations Bronze Table

This table stores traffic violations associated with accidents, such as speeding, DUI, and signal jumping.

In [0]:
violations_data = []

violation_types = ["Speeding", "DUI", "Signal Jump", "None"]

for i in range(1, 501):
    violations_data.append((
        i,
        random.randint(1, 300),
        random_string_case(random_choice(violation_types))
    ))

spark.sql("DROP TABLE IF EXISTS traffic_catalog.traffic_schema.violations_bronze")
violations_df = spark.createDataFrame(violations_data, ["violation_id", "accident_id", "violation_type"])
violations_df.write.format("delta").mode("overwrite").saveAsTable("traffic_catalog.traffic_schema.violations_bronze")

This table records traffic violations connected to each accident. It helps us later analyze which driver behaviors are most commonly associated with accidents.

### 2.7 Preview Bronze Tables

Before cleaning the data, we can inspect the raw Bronze tables to confirm that the mock data has been generated correctly.

In [0]:
display(spark.sql("SHOW TABLES IN traffic_catalog.traffic_schema"))
display(spark.table("traffic_catalog.traffic_schema.accidents_bronze").limit(10))
display(spark.table("traffic_catalog.traffic_schema.locations_bronze").limit(10))
display(spark.table("traffic_catalog.traffic_schema.weather_bronze").limit(10))
display(spark.table("traffic_catalog.traffic_schema.vehicles_bronze").limit(10))
display(spark.table("traffic_catalog.traffic_schema.violations_bronze").limit(10))

database,tableName,isTemporary
traffic_schema,accidents_bronze,false
traffic_schema,accidents_by_hour_gold,false
traffic_schema,accidents_silver,false
traffic_schema,age_group_risk_gold,false
traffic_schema,city_roadtype_gold,false
traffic_schema,hotspots_gold,false
traffic_schema,locations_bronze,false
traffic_schema,locations_silver,false
traffic_schema,road_risk_gold,false
traffic_schema,vehicles_bronze,false


accident_id,location_id,weather_id,vehicle_id,accident_date,accident_time,severity
1,74,38,11,2025-02-26,03:23:00,-1
2,48,39,76,2025-03-02,11:19:00,-1
3,76,54,13,2025-03-15,04:50:00,null
4,95,63,1,2025-04-21,05:52:00,1
5,6,54,124,2025-06-28,02:55:00,null
6,44,49,122,2025-10-17,14:57:00,-1
7,59,98,116,2025-03-12,16:40:00,-1
8,80,86,160,2025-06-16,02:51:00,null
9,70,6,172,2025-04-30,12:21:00,4
10,22,91,7,2025-05-23,20:05:00,5


location_id,city,road_type,speed_limit
1,burlington,highway,40
2,Toronto,urban,40
3,toronto,Highway,100
4,Toronto,Highway,100
5,mississauga,Urban,60
6,toronto,RURAL,60
7,burlington,HIGHWAY,40
8,TORONTO,HIGHWAY,60
9,burlington,RURAL,100
10,TORONTO,Highway,60


weather_id,condition,visibility,temperature
1,Fog,8.118952421910294,21.902279098410887
2,Clear,4.744105786216592,-15.70435028189079
3,Clear,9.077437522278183,9.51630870433063
4,fog,6.9628047604211725,6.885538175715432
5,fog,4.807606851707929,None
6,Fog,8.16569584926673,1.7771790474291613
7,Rain,1.1937804961968443,18.521168295344673
8,clear,9.86830385678163,-4.210011120977395
9,Rain,9.794166206778959,3.1199066385280325
10,FOG,8.76139271045173,0.6605043591093178


vehicle_id,vehicle_type,driver_age,driver_gender
1,TRUCK,-1,Male
2,Truck,-1,Male
3,TRUCK,null,Other
4,Truck,null,Other
5,BIKE,23,Female
6,Car,-1,Male
7,Truck,-1,Other
8,Bike,null,Male
9,TRUCK,null,Other
10,bike,-1,Female


violation_id,accident_id,violation_type
1,231,DUI
2,179,SIGNAL JUMP
3,214,SIGNAL JUMP
4,98,Speeding
5,267,Signal Jump
6,63,NONE
7,291,SIGNAL JUMP
8,57,SPEEDING
9,76,None
10,77,Signal Jump


## 3. Silver Layer — Data Cleaning and Standardization

The Silver layer improves data quality by cleaning the Bronze tables. In this stage, we are going to:
- standardize text formatting
- handle missing values
- correct data types
- remove invalid records

### 3.1 Read Bronze Tables

Now we are starting with loading all Bronze tables into Spark DataFrames for cleaning and transformation.

In [0]:
locations = spark.table("traffic_catalog.traffic_schema.locations_bronze")
weather = spark.table("traffic_catalog.traffic_schema.weather_bronze")
vehicles = spark.table("traffic_catalog.traffic_schema.vehicles_bronze")
accidents = spark.table("traffic_catalog.traffic_schema.accidents_bronze")
violations = spark.table("traffic_catalog.traffic_schema.violations_bronze")

### 3.2 Clean Locations Silver Table

In this step, we standardize city and road type values and remove invalid speed limits.

In [0]:
spark.sql("DROP TABLE IF EXISTS traffic_catalog.traffic_schema.locations_silver")

# Standardize text formatting and remove invalid speed limits
locations_silver = locations \
    .withColumn("city", initcap(col("city"))) \
    .withColumn("road_type", initcap(col("road_type"))) \
    .filter(col("speed_limit") > 0)

### 3.3 Clean Weather Silver Table

Here we standardize the weather condition text, convert temperature to the correct numeric type, and fill missing visibility values.

In [0]:
spark.sql("DROP TABLE IF EXISTS traffic_catalog.traffic_schema.weather_silver")

# Standardize weather condition names, cast temperature to numeric type,fill missing visibility values, and remove unrealistic temperatures
weather_silver = weather \
    .withColumn("condition", initcap(col("condition"))) \
    .withColumn("temperature", expr("try_cast(temperature as double)")) \
    .na.fill({"visibility": 5.0}) \
    .filter((col("temperature") >= -30) & (col("temperature") <= 50))

### 3.4 Clean Vehicles Silver Table

This step removes invalid driver ages and fills missing driver gender values.

In [0]:
spark.sql("DROP TABLE IF EXISTS traffic_catalog.traffic_schema.vehicles_silver")

# Clean vehicle type values, remove unrealistic driver ages, and replace missing gender with "Unknown"
vehicles_silver = vehicles \
    .withColumn("vehicle_type", initcap(col("vehicle_type"))) \
    .filter((col("driver_age") >= 16) & (col("driver_age") <= 90)) \
    .na.fill({"driver_gender": "Unknown"})

### 3.5 Clean Accidents Silver Table

This step removes invalid severity values so that only realistic accident severity levels remain.

In [0]:
spark.sql("DROP TABLE IF EXISTS traffic_catalog.traffic_schema.accidents_silver")

# Convert accident date and time into proper formats and keep only valid severity values
accidents_silver = accidents \
    .withColumn("accident_date", to_date(col("accident_date"), "yyyy-MM-dd")) \
    .withColumn("accident_time", to_timestamp(col("accident_time"), "HH:mm:ss")) \
    .filter((col("severity") >= 1) & (col("severity") <= 5)) \
    .filter(col("accident_date").isNotNull()) \
    .filter(col("accident_time").isNotNull())

### 3.6 Clean Violations Silver Table

This step standardizes violation type values for consistency across the dataset.

In [0]:
spark.sql("DROP TABLE IF EXISTS traffic_catalog.traffic_schema.violations_silver")

# Standardize violation labels and replace missing values
violations_silver = violations \
    .withColumn("violation_type", initcap(col("violation_type"))) \
    .na.fill({"violation_type": "None"})

### 3.7 Save Silver Tables

After cleaning, the transformed DataFrames are saved as Silver Delta tables. These Silver tables contain standardized and reliable data that can be used for joins, aggregation, and analysis in the Gold layer.

In [0]:
locations_silver.write.format("delta").mode("overwrite").saveAsTable("traffic_catalog.traffic_schema.locations_silver")
weather_silver.write.format("delta").mode("overwrite").saveAsTable("traffic_catalog.traffic_schema.weather_silver")
vehicles_silver.write.format("delta").mode("overwrite").saveAsTable("traffic_catalog.traffic_schema.vehicles_silver")
accidents_silver.write.format("delta").mode("overwrite").saveAsTable("traffic_catalog.traffic_schema.accidents_silver")
violations_silver.write.format("delta").mode("overwrite").saveAsTable("traffic_catalog.traffic_schema.violations_silver")

## 4. Gold Layer — Join and Aggregate Data

The Gold layer combines the cleaned Silver tables into business-ready analytical outputs. These tables are designed to support EDA, reporting, and dashboard creation.

In [0]:
display(spark.sql("SHOW TABLES IN traffic_catalog.traffic_schema"))

database,tableName,isTemporary
traffic_schema,accidents_bronze,false
traffic_schema,accidents_by_hour_gold,false
traffic_schema,accidents_silver,false
traffic_schema,age_group_risk_gold,false
traffic_schema,city_roadtype_gold,false
traffic_schema,hotspots_gold,false
traffic_schema,locations_bronze,false
traffic_schema,locations_silver,false
traffic_schema,road_risk_gold,false
traffic_schema,vehicles_bronze,false


### 4.1 Create Joined Gold DataFrame

We join the accidents, locations, weather, vehicles, and violations tables to create a single analytical DataFrame.

In [0]:
# Join the cleaned Silver tables to create a single analytics-ready DataFrame. This DataFrame combines accident, location, weather, vehicle, and violation details.
gold_df = accidents_silver \
    .join(locations_silver, "location_id") \
    .join(weather_silver, "weather_id") \
    .join(vehicles_silver, "vehicle_id") \
    .join(violations_silver, "accident_id", "left")\
    .withColumn("violation_type", coalesce(col("violation_type"), lit("None")))

### 4.2 Generate Accident Hotspots Gold Table

This Gold table summarizes accident counts by city. It helps to identify accident hotspot locations.

In [0]:
hotspots_gold = gold_df.groupBy("city") \
                                .count() \
                                .withColumnRenamed("count", "total_accidents") \
                                .orderBy(col("total_accidents").desc())

spark.sql("DROP TABLE IF EXISTS traffic_catalog.traffic_schema.hotspots_gold")
hotspots_gold.write.format("delta").mode("overwrite").saveAsTable(
                                                    "traffic_catalog.traffic_schema.hotspots_gold"
                                                )

### 4.3 Generate Weather Impact Gold Table

This Gold table measures average accident severity under different weather conditions. It helps analyze whether weather is linked to more severe accidents.


In [0]:
weather_impact_gold = gold_df.groupBy("condition") \
                                    .agg(round(avg("severity"), 2).alias("avg_severity")) \
                                    .orderBy(col("avg_severity").desc())

spark.sql("DROP TABLE IF EXISTS traffic_catalog.traffic_schema.weather_impact_gold")
weather_impact_gold.write.format("delta").mode("overwrite").saveAsTable(
                                                            "traffic_catalog.traffic_schema.weather_impact_gold"
                                                        )

### 4.4 Generate Violation Impact Gold Table

This Gold table shows how often each type of traffic violation appears in accident cases. It supports behavioral risk analysis.

In [0]:
violation_summary_gold = gold_df.groupBy("violation_type") \
                                        .count() \
                                        .withColumnRenamed("count", "total_cases") \
                                        .orderBy(col("total_cases").desc())

spark.sql("DROP TABLE IF EXISTS traffic_catalog.traffic_schema.violation_summary_gold")
violation_summary_gold.write.format("delta").mode("overwrite").saveAsTable(
                                                            "traffic_catalog.traffic_schema.violation_summary_gold"
                                                        )

### 4.5 Generate Driver Age Group Risk Gold Table
This Gold table groups drivers into age ranges and counts accident involvement. It helps compare accident frequency across age groups.

In [0]:
age_group_df = gold_df.withColumn(
                                    "age_group",
                                    when(col("driver_age") < 25, "Under 25")
                                    .when((col("driver_age") >= 25) & (col("driver_age") <= 40), "25-40")
                                    .when((col("driver_age") >= 41) & (col("driver_age") <= 60), "41-60")
                                    .otherwise("Above 60")
                                )

age_group_risk_gold = (
                            age_group_df
                            .groupBy("age_group")
                            .count()
                            .withColumnRenamed("count", "total_accidents")
                            .orderBy(col("total_accidents").desc())
                        )

spark.sql("DROP TABLE IF EXISTS traffic_catalog.traffic_schema.age_group_risk_gold")

age_group_risk_gold.write.format("delta").mode("overwrite").saveAsTable(
    "traffic_catalog.traffic_schema.age_group_risk_gold"
)

### 4.6 Generate Road Type Risk Gold Table
This Gold table compares average severity across different road types. It helps identify which road environments may be more dangerous.

In [0]:
road_risk_gold = gold_df.groupBy("road_type") \
                        .agg(round(avg("severity"), 2).alias("avg_severity")) \
                        .orderBy(col("avg_severity").desc())

spark.sql("DROP TABLE IF EXISTS traffic_catalog.traffic_schema.road_risk_gold")
road_risk_gold.write.format("delta").mode("overwrite").saveAsTable(
                                                    "traffic_catalog.traffic_schema.road_risk_gold"
                                                )

### 4.7 Generate Accidents by Hour Gold Table
This Gold table summarizes accident frequency by hour of day. It supports time-based trend analysis.

In [0]:
accidents_with_hour_gold = accidents_silver.withColumn(
                                                        "accident_hour",
                                                        hour(col("accident_time"))
                                                    )

accidents_by_hour_gold = (
                            accidents_with_hour_gold
                            .groupBy("accident_hour")
                            .count()
                            .withColumnRenamed("count", "accident_count")
                            .orderBy("accident_hour")
                        )

spark.sql("DROP TABLE IF EXISTS traffic_catalog.traffic_schema.accidents_by_hour_gold")

accidents_by_hour_gold.write.format("delta").mode("overwrite").saveAsTable(
    "traffic_catalog.traffic_schema.accidents_by_hour_gold"
)

### 4.8 Generate Accidents by City and Road Type Gold Table
This Gold table combines city and road type to identify detailed accident concentration patterns. It is useful for dashboard heatmaps or stacked bar charts.

In [0]:
city_roadtype_gold = gold_df.groupBy("city", "road_type").count()\
.withColumnRenamed("count", "total_accidents")\
.orderBy(col("total_accidents").desc())

spark.sql("DROP TABLE IF EXISTS traffic_catalog.traffic_schema.city_roadtype_gold")
city_roadtype_gold.write.format("delta").mode("overwrite").saveAsTable( "traffic_catalog.traffic_schema.city_roadtype_gold" )

### 4.9 Generate Severity Distribution Gold Table
This table shows how accidents are spread across severity levels 1–5

In [0]:

severity_distribution_gold = gold_df.groupBy("severity") \
                                    .count() \
                                    .withColumnRenamed("count", "total_accidents") \
                                    .orderBy("severity")

spark.sql("DROP TABLE IF EXISTS traffic_catalog.traffic_schema.severity_distribution_gold")
severity_distribution_gold.write.format("delta").mode("overwrite").saveAsTable(
    "traffic_catalog.traffic_schema.severity_distribution_gold"
)

## 4.9 Save Gold tables

The following Gold tables can be used directly in Databricks dashboard visualizations.

In [0]:
display(spark.sql("SHOW TABLES IN traffic_catalog.traffic_schema"))

database,tableName,isTemporary
traffic_schema,accidents_bronze,false
traffic_schema,accidents_by_hour_gold,false
traffic_schema,accidents_silver,false
traffic_schema,age_group_risk_gold,false
traffic_schema,city_roadtype_gold,false
traffic_schema,hotspots_gold,false
traffic_schema,locations_bronze,false
traffic_schema,locations_silver,false
traffic_schema,road_risk_gold,false
traffic_schema,severity_distribution_gold,false


In [0]:
display(spark.table("traffic_catalog.traffic_schema.hotspots_gold"))
display(spark.table("traffic_catalog.traffic_schema.weather_impact_gold"))
display(spark.table("traffic_catalog.traffic_schema.violation_summary_gold"))
display(spark.table("traffic_catalog.traffic_schema.age_group_risk_gold"))
display(spark.table("traffic_catalog.traffic_schema.road_risk_gold"))
display(spark.table("traffic_catalog.traffic_schema.accidents_by_hour_gold"))
display(spark.table("traffic_catalog.traffic_schema.city_roadtype_gold"))

city,total_accidents
Mississauga,28
Toronto,26
Burlington,22
Brampton,12


condition,avg_severity
Clear,3.35
Rain,2.91
Fog,2.75
Snow,1.75


violation_type,total_cases
None,39
Speeding,19
Dui,16
Signal Jump,14


age_group,total_accidents
Above 60,34
25-40,26
Under 25,17
41-60,11


road_type,avg_severity
Highway,2.97
Rural,2.92
Urban,2.83


accident_hour,accident_count
0,8
1,6
2,6
3,4
4,5
5,6
6,7
7,3
8,6
9,6


city,road_type,total_accidents
Mississauga,Urban,12
Toronto,Rural,12
Mississauga,Rural,12
Toronto,Highway,12
Burlington,Highway,11
Burlington,Rural,9
Brampton,Highway,5
Brampton,Rural,5
Mississauga,Highway,4
Burlington,Urban,2


## 5. Exploratory Data Analysis (EDA)

In this section, we perform simple exploratory data analysis using the Gold tables. The goal is to identify initial trends and patterns that will later be presented more clearly in the dashboard.

### 5.1 Descriptive Statistics

We first review summary statistics from the joined Gold DataFrame to better understand the overall dataset.

In [0]:
gold_df.describe().show()

+-------+------------------+------------------+------------------+------------------+------------------+--------+---------+------------------+---------+------------------+-------------------+------------+------------------+-------------+------------------+--------------+
|summary|       accident_id|        vehicle_id|        weather_id|       location_id|          severity|    city|road_type|       speed_limit|condition|        visibility|        temperature|vehicle_type|        driver_age|driver_gender|      violation_id|violation_type|
+-------+------------------+------------------+------------------+------------------+------------------+--------+---------+------------------+---------+------------------+-------------------+------------+------------------+-------------+------------------+--------------+
|  count|                88|                88|                88|                88|                88|      88|       88|                88|       88|                88|             

### 5.2 Accident Hotspots by City

This output highlights the cities with the highest number of recorded accidents.

In [0]:
display(hotspots_gold)

city,total_accidents
Mississauga,28
Toronto,26
Burlington,22
Brampton,12


Observation: Some cities have noticeably higher accident counts than others, suggesting possible high-risk urban areas.

### 5.3 Weather Condition vs. Severity

This output helps us understand whether weather conditions are associated with higher accident severity.

In [0]:
display(weather_impact_gold)

condition,avg_severity
Clear,3.35
Rain,2.91
Fog,2.75
Snow,1.75


Observation: Adverse weather conditions such as rain, snow, or fog tend to show higher average accident severity than clear weather.

### 5.4 Violation Type Distribution

This output compares how frequently each traffic violation appears in the accident dataset.

In [0]:
display(violation_summary_gold)

violation_type,total_cases
None,39
Speeding,19
Dui,16
Signal Jump,14


Observation: Certain traffic violations appear more frequently in accident records, indicating that driver behavior is an important risk factor.

## 6. Dashboard-Ready Outputs

In this final section, we reload the Gold tables that will be used directly in the Databricks dashboard. These tables are already aggregated and optimized for visual analysis. The dashboard will focus on accident hotspots, severity drivers, behavioral risk, and time-based trends.

### 6.1 Load Gold Tables for Dashboard Use

We are reloading the saved Gold tables so they can be used directly in dashboard creation or plotting.

In [0]:
hotspots = spark.table("traffic_catalog.traffic_schema.hotspots_gold")
weather_impact = spark.table("traffic_catalog.traffic_schema.weather_impact_gold")
violation_summary = spark.table("traffic_catalog.traffic_schema.violation_summary_gold")
age_group_risk = spark.table("traffic_catalog.traffic_schema.age_group_risk_gold")
road_risk = spark.table("traffic_catalog.traffic_schema.road_risk_gold")
accidents_by_hour = spark.table("traffic_catalog.traffic_schema.accidents_by_hour_gold")
city_roadtype_risk = spark.table("traffic_catalog.traffic_schema.city_roadtype_gold")

### 6.2 Accident Hotspots by City

This output will be used to create a Databricks bar chart showing the cities with the highest accident counts.

In [0]:
display(hotspots_gold)

city,total_accidents
Mississauga,28
Toronto,26
Burlington,22
Brampton,12


### 6.3 Weather Condition vs. Severity

This output will be used to create a Databricks bar chart comparing average accident severity across weather conditions.

In [0]:
display(weather_impact_gold)

condition,avg_severity
Clear,3.35
Rain,2.91
Fog,2.75
Snow,1.75


### 6.4 Violation Type Analysis

This output will be used to create a Databricks chart showing the distribution of violation types linked to accidents.

In [0]:
display(violation_summary_gold)

violation_type,total_cases
None,39
Speeding,19
Dui,16
Signal Jump,14


### 6.5 Driver Age Group Risk

This output supports a dashboard chart showing which age groups are most frequently involved in accidents.

In [0]:
display(age_group_risk_gold)

age_group,total_accidents
Above 60,34
25-40,26
Under 25,17
41-60,11


### 6.6 Road Type Risk Analysis

This output supports a dashboard chart comparing average accident severity across different road types.

In [0]:
display(road_risk_gold)
display(accidents_by_hour_gold)
display(city_roadtype_gold)

road_type,avg_severity
Highway,2.97
Rural,2.92
Urban,2.83


accident_hour,accident_count
0,8
1,6
2,6
3,4
4,5
5,6
6,7
7,3
8,6
9,6


city,road_type,total_accidents
Mississauga,Urban,12
Toronto,Rural,12
Mississauga,Rural,12
Toronto,Highway,12
Burlington,Highway,11
Burlington,Rural,9
Brampton,Highway,5
Brampton,Rural,5
Mississauga,Highway,4
Burlington,Urban,2


## 7. Conclusion

This notebook demonstrates a complete big data pipeline using Databricks and PySpark. We generated raw traffic accident data in the Bronze layer, cleaned and standardized it in the Silver layer, and created business-ready Gold tables for analysis. The results show that accident risk is influenced by location, weather, road type, driver age, violations, and time of day. These outputs are used to build the final dashboard for presentation.